In [ ]:
# !pip install cyvcf2

In [ ]:
# CÉLULA DE TESTE DA BIBLIOTECA CYVCF2

from cyvcf2 import VCF

FILE_NAME = 'Y.vcf'
FILE_PATH = f'/home/marcela/IC/IC/files/{FILE_NAME}'

# Abre o arquivo .vcf (ou .vcf.gz)
vcf = VCF(FILE_PATH)

#Caso se queira visualizar somente o header
#print(vcf.raw_header)

#visualização das amostras participantes
# print("amostras: ", vcf.samples)

#Iterando sobre as variantes(linhas do Vcf)
for variant in vcf:

  #Impressão da linha completa
  print("Variante: ", variant)
  
  #Leitura de dados básicos
  cromossomo = variant.CHROM
  posicao = variant.POS
  id = variant.ID
  ref = variant.REF
  alt = variant.ALT
  quality = variant.QUAL

  #Acessando o campo INFO
  dp = variant.INFO.get('DP')
  ac = variant.INFO.get('AC')
  af = variant.INFO.get('AF')

  #Acessando os genótipos das amostras
  genotipos = variant.genotypes
  gt_types = variant.gt_types
  gt_ref_depths = variant.gt_ref_depths
  gt_alt_depths = variant.gt_alt_depths
  gt_phases = variant.gt_phases
  gt_quals = variant.gt_quals
  gt_bases = variant.gt_bases

  # print(f'CHROM: {cromossomo}\nPOS: {posicao}\nID: {id}\nREF: {ref}\nALT: {alt}\nQUAL: {quality:.2f}\n')
  print(genotipos)
  # print(gt_types)
  # print(gt_alt_depths)
  # print(gt_ref_depths)
  # print(gt_phases)
  # print(gt_quals)
  # print(gt_bases)

  #Forma de acessar informações especificas do campo FORMAT (para uma determinada amostra deve-se especificar o idx) 
  # (ex: os resultados de DP daquela amostra em todas as variantes)
  sample_idx = 1
  dp_array = variant.format('DP')
  dp = dp_array[sample_idx].tolist() if dp_array is not None else None
  dp_val = dp_array[sample_idx][0] if dp_array is not None else "."

  print(dp)
  print(dp_val)

  #Para imprimir apenas o primeiro
  break

vcf.close()


In [ ]:
from cyvcf2 import VCF
import pandas as pd
import os
from enum import Enum


# FILE_NAME = 'Y.vcf'
FILE_NAME = 'Cyberseg_chr21.vcf'
FILE_PATH = f'/home/marcela/IC/IC/files/{FILE_NAME}'

VCF_FILE = VCF(FILE_PATH)

class BASIC_COLS(Enum):
    CHROM = 0
    POS = 1
    ID = 2
    REF = 3
    ALT = 4
    QUAL = 5
    FILTER = 6
    INFO = 7  
    FORMAT = 8


def vcf_to_df_raw(vcf_path, vcf_file):
    cmd = "zgrep '^#' " + vcf_path + "|tail -n 1"
    cols = os.popen(cmd).read().strip('#').strip('\n').split('\t')
    
    data = []
    
    for variant in vcf_file:
        raw_line = str(variant).rstrip('\n').split('\t')
        data.append(raw_line)

    df = pd.DataFrame(data, columns=cols)
    return df

def vcf_to_df_filtered_INFO(vcf_file):
    #Define as colunas desejadas e define subcolunas para pegar apenas parte de INFO
    cols_tuples = [
        ('#CHROM', ''),
        ('POS', ''),
        ('REF', ''),
        ('ALT', ''),
        ('INFO', 'AC'),  
        ('INFO', 'AF'),  
        ('INFO', 'DP'),  
        ('FORMAT', '')
    ]

    #Acrescenta as colunas de amostras
    for sample in vcf_file.samples:
        cols_tuples.append((sample, ''))

    multi_cols = pd.MultiIndex.from_tuples(cols_tuples)

    data = []
    for variant in vcf_file:
        raw_line = str(variant).strip('\n').split('\t')

        fltrd_line = [
        	raw_line[BASIC_COLS.CHROM.value],
            raw_line[BASIC_COLS.POS.value],
            raw_line[BASIC_COLS.REF.value],
            raw_line[BASIC_COLS.ALT.value],
            variant.INFO.get('AC'),
            variant.INFO.get('AF'),
            variant.INFO.get('DP'),
            raw_line[BASIC_COLS.FORMAT.value] #PROVISORIO! depois filtar o format tbm (pegar só os 4 prieiros)
        ]

        fltrd_line.extend(raw_line[BASIC_COLS.FORMAT.value+1:])
        data.append(fltrd_line)
    
    df = pd.DataFrame(data, columns=multi_cols)
    return df

def vcf_to_df_filtered_Samples(vcf_file):
    #Define as colunas desejadas e define subcolunas para pegar apenas parte de INFO
    cols_tuples = [
        ('#CHROM', ''),
        ('POS', ''),
        ('REF', ''),
        ('ALT', ''),
        ('INFO', 'AC'),  
        ('INFO', 'AF'),  
        ('INFO', 'DP'),  
    ]

    #Acrescenta as colunas de amostras
    for sample in vcf_file.samples:
        sample_tuple = [
            (sample, 'GT'),
            (sample, 'AF'),
            (sample, 'DP'),
        ]
        cols_tuples.extend(sample_tuple)

    multi_cols = pd.MultiIndex.from_tuples(cols_tuples)

    data = []
    for variant in vcf_file:
        raw_line = str(variant).strip('\n').split('\t')

        fltrd_line = [
        	raw_line[BASIC_COLS.CHROM.value],
            raw_line[BASIC_COLS.POS.value],
            raw_line[BASIC_COLS.REF.value],
            raw_line[BASIC_COLS.ALT.value],
            variant.INFO.get('AC'),
            variant.INFO.get('AF'),
            variant.INFO.get('DP'),
        ]

        # Extração segura com cyvcf2. Retorna arrays ou None.
        genotypes = variant.genotypes
        af_array = variant.format('AF')
        dp_array = variant.format('DP')

        samples_fltrd_line = []
        for i in range(len(vcf_file.samples)):
            #tratamento de genotypes
   
            # genotypes[i] retorna algo como [0, 1, False].
            # Tratamento do GT (suporta diploides e haploides)
            if genotypes is not None and len(genotypes[i]) > 0:
                gt_sample = genotypes[i]
                # O último elemento [-1] é sempre o booleano indicando o phase
                phased = gt_sample[-1] 
                
                # Pega todos os elementos do início até o penúltimo [:-1] (os alelos em si)
                alelos = gt_sample[:-1] 
                
                sep = "|" if phased else "/"
                
                # Transforma -1 em "." e converte os números para string
                alelos_str = [str(a) if a != -1 else "." for a in alelos]
                
                # Junta os alelos com a barra ou pipe correspondente
                gt_val = sep.join(alelos_str)
            else:
                gt_val = "./."
                
            # Pegamos o valor se o array existir, senão colocamos "."
            af_val = af_array[i][0] if af_array is not None else "."
            dp_val = dp_array[i][0] if dp_array is not None else "."

            sample_fltrd = [
                gt_val,
                af_val,
                dp_val
            ]
            
            samples_fltrd_line.extend(sample_fltrd)
        
        fltrd_line.extend(samples_fltrd_line)
        data.append(fltrd_line)
    
    df = pd.DataFrame(data, columns=multi_cols)
    return df
    

# df_raw = vcf_to_df_raw(FILE_PATH, VCF_FILE)
# df_raw

# df_filtered_INFO = vcf_to_df_filtered_INFO(VCF_FILE)
# df_filtered_INFO.head()

df_filtered_Samples = vcf_to_df_filtered_Samples(VCF_FILE)
df_filtered_Samples.head()

# VCF_FILE.close()



# filtragem por genotipo e afins
# Sugestão de melhoria: para arquivos muitos grandes, vale a pena fazer o processamento com chunks